# Module 09: Pandas for Data Science & Machine Learning

**Duration:** 30 hours  
**ML Focus:** Titanic Dataset EDA & Data Preparation

Pandas is the fundamental data manipulation library in Python. This lesson covers everything from basic Series/DataFrame operations to advanced techniques, all with a focus on preparing data for machine learning workflows.

## 1. Series and DataFrame Fundamentals

A **Series** is a 1D labeled array. A **DataFrame** is a 2D labeled data structure with columns of potentially different types.

In [ ]:
import pandas as pd
import numpy as np

# Creating Series
s1 = pd.Series([1, 2, 3, 4, 5], name='numbers')
s2 = pd.Series({'a': 10, 'b': 20, 'c': 30})
print('Series from list:', s1.values)
print('Series from dict:', s2.index.tolist())

# Creating DataFrames
df = pd.DataFrame({
    'Name': ['Alice', 'Bob', 'Charlie', 'Diana'],
    'Age': [25, 32, 37, 29],
    'Salary': [70000, 85000, 95000, 72000],
    'Department': ['IT', 'HR', 'IT', 'Finance']
})
print('\nDataFrame shape:', df.shape)
print('DataFrame columns:', df.columns.tolist())
print('DataFrame dtypes:')
print(df.dtypes)

## 2. Reading Data from Various Sources

Pandas can read from CSV, Excel, SQL, JSON, Parquet, and more. We'll load the Titanic dataset.

In [ ]:
# Load Titanic dataset from URL
url = 'https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv'
titanic = pd.read_csv(url)

# Quick inspection
print('Shape:', titanic.shape)
print('\nFirst 5 rows:')
print(titanic.head())

print('\nColumn names:', titanic.columns.tolist())
print('\nBasic info:')
print(titanic.info())

## 3. Data Selection with .loc, .iloc, and Boolean Indexing

Mastering data selection is critical for efficient data wrangling.

In [ ]:
print('=== loc: label-based ===')
print(titanic.loc[0:3, ['Name', 'Age', 'Survived']])

print('\n=== iloc: position-based ===')
print(titanic.iloc[0:3, [0, 2, 3, 5]])

print('\n=== Boolean indexing ===')
survived_males = titanic.loc[(titanic['Survived'] == 1) & (titanic['Sex'] == 'male')]
print(f'Number of survived males: {len(survived_males)}')
print(survived_males[['Name', 'Age', 'Pclass']].head())

print('\n=== Chained vs. non-chained (avoid this): ===')
# Bad: df[df['col'] > 0]['other']  -- SettingWithCopyWarning risk
# Good: df.loc[df['col'] > 0, 'other']

## 4. Handling Missing Data

Real-world datasets always have missing values. Proper handling is essential for ML model performance.

In [ ]:
print('=== Missing value count ===')
print(titanic.isna().sum())

print('\n=== Missing value percentage ===')
print((titanic.isna().sum() / len(titanic) * 100).round(2))

# Fill missing Age with median grouped by Pclass and Sex
age_medians = titanic.groupby(['Pclass', 'Sex'])['Age'].median()
print('\nAge medians by Pclass and Sex:')
print(age_medians)

# Apply fillna with transform for group-aware imputation
titanic['Age'] = titanic.groupby(['Pclass', 'Sex'])['Age'].transform(
    lambda x: x.fillna(x.median())
)
print(f'\nMissing Age after imputation: {titanic["Age"].isna().sum()}')

# Fill Embarked with mode
titanic['Embarked'] = titanic['Embarked'].fillna(titanic['Embarked'].mode()[0])
print(f'Missing Embarked after: {titanic["Embarked"].isna().sum()}')

# Interpolate example (for numeric sequential data)
demo_series = pd.Series([1, np.nan, np.nan, 4, 5, np.nan, 7])
print('\nInterpolation demo:', demo_series.interpolate().values)

## 5. GroupBy Operations

The Split-Apply-Combine pattern is central to pandas. GroupBy allows aggregation, transformation, and filtering by groups.

In [ ]:
print('=== GroupBy aggregation ===')
grouped = titanic.groupby('Pclass')['Fare'].agg(['mean', 'median', 'std', 'count'])
print(grouped)

print('\n=== Multiple aggregations per column ===')
result = titanic.groupby(['Pclass', 'Sex']).agg({
    'Fare': ['mean', 'max'],
    'Age': ['mean', 'median'],
    'Survived': 'mean'
}).round(2)
print(result)

print('\n=== Transform: center values within groups ===')
titanic['Fare_centered'] = titanic.groupby('Pclass')['Fare'].transform(
    lambda x: x - x.mean()
)
print(titanic[['Pclass', 'Fare', 'Fare_centered']].head())

print('\n=== Filter: keep groups with condition ===')
# Keep Pclass groups where max Fare > 200
filtered = titanic.groupby('Pclass').filter(lambda x: x['Fare'].max() > 200)
print(f'Rows after filter: {len(filtered)} (original: {len(titanic)})')

## 6. Merge, Join, Concatenate

Combining datasets is essential when working with relational data.

In [ ]:
# Create sample related datasets
passengers = titanic[['PassengerId', 'Name', 'Pclass', 'Survived']].head(10)
ticket_info = pd.DataFrame({
    'PassengerId': range(1, 11),
    'TicketClass': ['First'] * 3 + ['Second'] * 3 + ['Third'] * 4,
    'Luggage': np.random.randint(0, 3, 10)
})

print('=== Inner merge ===')
merged = pd.merge(passengers, ticket_info, on='PassengerId', how='inner')
print(merged.head())

print('\n=== Concatenation (row-wise) ===')
df_a = titanic.head(3)
df_b = titanic.iloc[3:6]
concat_rows = pd.concat([df_a, df_b])
print(f'Concat rows: {concat_rows.shape[0]} rows (3+3)')

print('\n=== Join on index ===')
left = passengers.set_index('PassengerId')
right = ticket_info.set_index('PassengerId')
joined = left.join(right, how='left')
print(joined.head())

## 7. Apply / Map / ApplyMap

Vectorized operations are preferred, but `.apply()` and `.map()` are powerful for custom transformations.

In [ ]:
print('=== .map() on Series ===')
titanic['Sex_binary'] = titanic['Sex'].map({'male': 0, 'female': 1})
print('Sex -> binary mapping:')
print(titanic[['Sex', 'Sex_binary']].head())

print('\n=== .apply() on Series ===')
def age_category(age):
    if age < 12:
        return 'Child'
    elif age < 20:
        return 'Teen'
    elif age < 60:
        return 'Adult'
    else:
        return 'Senior'

titanic['Age_Group'] = titanic['Age'].apply(age_category)
print(titanic['Age_Group'].value_counts())

print('\n=== .apply() on DataFrame (row-wise) ===')
titanic['Family_Size'] = titanic.apply(
    lambda row: row['SibSp'] + row['Parch'] + 1, axis=1
)
print('Family size distribution:')
print(titanic['Family_Size'].value_counts().sort_index())

# Note: .applymap() is deprecated in pandas 2.1+; use .map() on DataFrames

## 8. Pivot Tables, Datetime, and String Operations

These three topics are essential for real-world data analysis workflows.

In [ ]:
print('=== Pivot Table: Survival rates ===')
pivot = pd.pivot_table(
    titanic, values='Survived', index='Pclass', columns='Sex',
    aggfunc='mean', margins=True
)
print(pivot.round(3))

print('\n=== Crosstab ===')
ct = pd.crosstab(titanic['Pclass'], titanic['Survived'], normalize='index')
print(ct.round(3))

print('\n=== Datetime operations ===')
dates = pd.date_range('2024-01-01', periods=5, freq='D')
demo_dates = pd.DataFrame({'date': dates, 'value': range(5)})
demo_dates['year'] = demo_dates['date'].dt.year
demo_dates['month'] = demo_dates['date'].dt.month
demo_dates['weekday'] = demo_dates['date'].dt.day_name()
print(demo_dates)

print('\n=== String operations ===')
# Extract title from Name column
titanic['Title'] = titanic['Name'].str.extract(r',\s*([^\.]+)\.', expand=False)
print('Title distribution:')
print(titanic['Title'].value_counts())

# Extract Cabin deck letter
titanic['Deck'] = titanic['Cabin'].str.extract(r'([A-Z])', expand=False)
print('\nDeck distribution:')
print(titanic['Deck'].value_counts(dropna=False))

## 9. Memory Optimization and Multi-Index

Efficient memory usage becomes critical with large datasets. Multi-indexing enables hierarchical data organization.

In [ ]:
print('=== Memory optimization ===')
print('Before optimization:')
print(titanic.memory_usage(deep=True))

# Convert object columns with low cardinality to category
for col in ['Sex', 'Embarked', 'Title', 'Deck', 'Age_Group', 'Pclass']:
    if col in titanic.columns:
        titanic[col] = titanic[col].astype('category')

print('\nAfter categorical conversion:')
print(titanic.memory_usage(deep=True))

print('\n=== Multi-Index ===')
mi_df = titanic.set_index(['Pclass', 'Sex']).sort_index()
print('Multi-index DataFrame (head):')
print(mi_df[['Age', 'Fare', 'Survived']].head())

print('\nSelecting using .xs():')
class1_females = mi_df.xs((1, 'female'), level=['Pclass', 'Sex'])
print(f'Class 1, Female passengers: {len(class1_females)}')
print(class1_females[['Age', 'Fare', 'Survived']].head())

## 10. Preparing the Titanic Dataset for ML

Final step: convert the cleaned dataset into a numeric feature matrix ready for scikit-learn.

In [ ]:
print('=== Final ML-ready dataset ===')

# Select features for modeling
ml_features = titanic[['Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare',
                       'Embarked', 'Title', 'Family_Size']].copy()

# Encode categorical features
ml_features = pd.get_dummies(ml_features, columns=['Sex', 'Embarked', 'Title'],
                             drop_first=True, dtype=int)

# Target
target = titanic['Survived']

print('Feature matrix shape:', ml_features.shape)
print('\nFeature columns:')
print(ml_features.columns.tolist())
print('\nTarget distribution:')
print(target.value_counts(normalize=True).round(3))
print('\nMissing values in features:', ml_features.isna().sum().sum())
print('\nFeature matrix preview:')
print(ml_features.head())

print('\n=== Summary ===')
print('Module 09 complete! You now have:')
print('  - A clean, imputed Titanic dataset')
print('  - Engineered features (Family_Size, Title, Age_Group)')
print('  - Properly encoded categorical variables')
print('  - Memory-optimized data types')
print('  - A feature matrix ready for ML modeling')